In [ ]:
from __future__ import annotations

import os
import warnings
from typing import Dict, Iterable, List, Mapping, Tuple, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from catboost import CatBoostClassifier, Pool

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (
    precision_recall_curve,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    roc_auc_score,
    roc_curve,
    auc,
    brier_score_loss,
)
from sklearn.calibration import calibration_curve, CalibratedClassifierCV
from sklearn.linear_model import LogisticRegression

import optuna

warnings.filterwarnings("ignore")


data_fs_static_external = pd.read_csv("YOUR_PATH")
data_fs_static_train = pd.read_csv("YOUR_PATH")

scale = 'yes'
feature_space = ['feature_1', 'feature_2', ...]

X_external, y_external  = do_train_test_split(data_fs_static_external,feature_space,scale)
X_train, y_train  = do_train_test_split(data_fs_static_train,feature_space,scale)


SUBJECT_COL = "subject_reference"

def _align_subject_reference(
    raw_df: pd.DataFrame,
    X: pd.DataFrame,
    subject_col: str = SUBJECT_COL,
) -> np.ndarray:
    """Align patient identifiers to the exact rows/order returned by preprocessing."""
    if subject_col not in raw_df.columns:
        raise KeyError(f"Required grouping column '{subject_col}' not found in raw dataframe.")
    if subject_col in X.columns:
        raise ValueError(
            f"Grouping column '{subject_col}' must not be included among model predictors."
        )
    if not hasattr(X, "index"):
        raise TypeError("X must be a pandas DataFrame with an index for safe patient-ID alignment.")

    missing_idx = X.index.difference(raw_df.index)
    if len(missing_idx) == 0:
        groups = raw_df.loc[X.index, subject_col].to_numpy()
    else:
        raise ValueError(
            "Could not safely align subject_reference to processed feature rows because "
            "X contains indices not present in the source dataframe. Modify do_train_test_split() "
            "to preserve the source dataframe index."
        )

    if len(groups) != len(X):
        raise RuntimeError("Patient-ID alignment produced a length mismatch.")
    if pd.isna(groups).any():
        raise ValueError(f"Missing values found in grouping column '{subject_col}'.")
    return np.asarray(groups)

subject_reference_train = _align_subject_reference(data_fs_static_train, X_train)
subject_reference_external = _align_subject_reference(data_fs_static_external, X_external)

print(
    f"Training: {len(X_train)} admissions from "
    f"{pd.Series(subject_reference_train).nunique()} unique patients."
)
print(
    f"External: {len(X_external)} admissions from "
    f"{pd.Series(subject_reference_external).nunique()} unique patients."
)


In [ ]:
RANDOM_SEED = 42

# Optuna / CV
N_TRIALS = 200
N_SPLITS_INNER = 5          # CV for Optuna objective
N_SPLITS_THRESHOLD = 5      # CV for OOF threshold selection

# Bootstrap
N_BOOTSTRAPS = 2000

# Outputs
FIG_DIR = "figures"
SHAP_DIR = "shap_values"
OUT_DIR = "outputs"
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(SHAP_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)

plt.rcParams.update(
    {
        "figure.dpi": 140,
        "savefig.dpi": 300,
        "font.size": 11,
        "axes.titlesize": 12,
        "axes.labelsize": 11,
        "legend.fontsize": 10,
        "axes.grid": True,
        "grid.alpha": 0.25,
    }
)

print("Ready.")

def as_numpy(y: Iterable) -> np.ndarray:
    y_arr = np.asarray(y)
    return y_arr.reshape(-1)

def safe_confusion(yt: np.ndarray, yp: np.ndarray) -> Tuple[int, int, int, int]:
    cm = confusion_matrix(yt, yp, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    return int(tn), int(fp), int(fn), int(tp)

def ci_mean(vals: Iterable[float], alpha: float = 0.05) -> Tuple[float, float, float]:
    v = np.asarray(list(vals), dtype=float)
    lo = np.percentile(v, 100 * (alpha / 2))
    hi = np.percentile(v, 100 * (1 - alpha / 2))
    return float(v.mean()), float(lo), float(hi)

def stable_roc_auc(y_true: np.ndarray, y_prob: np.ndarray) -> float:
    if len(np.unique(y_true)) < 2:
        return float("nan")
    return float(roc_auc_score(y_true, y_prob))

def stable_auprc(y_true: np.ndarray, y_prob: np.ndarray) -> float:
    if len(np.unique(y_true)) < 2:
        return float("nan")
    prec, rec, _ = precision_recall_curve(y_true, y_prob)
    return float(auc(rec, prec))

def summarize_bootstrap(boot: Mapping[str, List[float]]) -> pd.DataFrame:
    rows = []
    for k in ["accuracy","precision","recall","specificity","sensitivity","npv","f1","mcc","auc","auprc"]:
        vals = [v for v in boot[k] if not np.isnan(v)]
        m, lo, hi = ci_mean(vals)
        rows.append([k, m, lo, hi])
    return pd.DataFrame(rows, columns=["metric", "mean", "ci_low", "ci_high"])

def align_features(
    X_target: pd.DataFrame,
    X_reference: pd.DataFrame,
    fill_strategy: str = "mean",  # "mean" or "zero"
) -> pd.DataFrame:
    X_aligned = X_target.copy()
    missing = [c for c in X_reference.columns if c not in X_aligned.columns]
    if missing:
        if fill_strategy == "mean":
            fill_vals = X_reference[missing].mean()
            for c in missing:
                X_aligned[c] = float(fill_vals[c])
        elif fill_strategy == "zero":
            for c in missing:
                X_aligned[c] = 0.0
        else:
            raise ValueError(f"Unknown fill_strategy='{fill_strategy}'")
    return X_aligned.reindex(columns=X_reference.columns)

def thresholds_from_predictions(y_true: np.ndarray, y_prob: np.ndarray) -> Dict[str, float]:
    y_true = as_numpy(y_true)
    y_prob = as_numpy(y_prob)

    prec, rec, thresh = precision_recall_curve(y_true, y_prob)
    if len(thresh) == 0:
        return {"F1-optimal": 0.5, "MCC-optimal": 0.5, "Youden": 0.5}

    f1_vals = 2 * (prec * rec) / (prec + rec + 1e-12)
    t_f1 = float(thresh[int(np.nanargmax(f1_vals[:-1]))])

    mcc_vals = [matthews_corrcoef(y_true, (y_prob >= t).astype(int)) for t in thresh]
    t_mcc = float(thresh[int(np.nanargmax(mcc_vals))])

    youden_vals = []
    for t in thresh:
        yp = (y_prob >= t).astype(int)
        tn, fp, fn, tp = safe_confusion(y_true, yp)
        sens = tp / (tp + fn + 1e-12)
        spec = tn / (tn + fp + 1e-12)
        youden_vals.append(sens + spec - 1)
    t_youden = float(thresh[int(np.nanargmax(youden_vals))])

    return {"F1-optimal": t_f1, "MCC-optimal": t_mcc, "Youden": t_youden}

def _cluster_bootstrap_indices(
    groups: np.ndarray,
    rng: np.random.RandomState,
) -> np.ndarray:
    """Sample patients with replacement and retain all admissions for each sampled patient."""
    groups = np.asarray(groups)
    unique_groups = pd.unique(groups)
    if len(unique_groups) == 0:
        raise ValueError("No patient groups available for bootstrap.")
    sampled_groups = rng.choice(unique_groups, size=len(unique_groups), replace=True)
    return np.concatenate([np.flatnonzero(groups == g) for g in sampled_groups])


def bootstrap_metrics_fixed_threshold(
    y_true: np.ndarray,
    y_prob: np.ndarray,
    groups: Iterable,
    threshold: float,
    n_boot: int = N_BOOTSTRAPS,
    seed: int = RANDOM_SEED,
) -> Dict[str, List[float]]:
    """Patient-cluster bootstrap; metrics remain admission-level."""
    rng = np.random.RandomState(seed)
    y_true = as_numpy(y_true)
    y_prob = as_numpy(y_prob)
    g_np = np.asarray(groups)

    if not (len(y_true) == len(y_prob) == len(g_np)):
        raise ValueError("y_true, y_prob, and groups must have identical lengths.")

    keys = ["accuracy","precision","recall","specificity","sensitivity","npv",
            "f1","mcc","auc","auprc","tn","fp","fn","tp"]
    M: Dict[str, List[float]] = {k: [] for k in keys}

    for _ in range(n_boot):
        idx = _cluster_bootstrap_indices(g_np, rng)
        yt = y_true[idx]
        pr = y_prob[idx]
        yp = (pr >= threshold).astype(int)

        tn, fp, fn, tp = safe_confusion(yt, yp)
        spec = tn / (tn + fp + 1e-12)
        sens = tp / (tp + fn + 1e-12)
        npv  = tn / (tn + fn + 1e-12)

        M["accuracy"].append(accuracy_score(yt, yp))
        M["precision"].append(precision_score(yt, yp, zero_division=0))
        M["recall"].append(recall_score(yt, yp, zero_division=0))
        M["specificity"].append(spec)
        M["sensitivity"].append(sens)
        M["npv"].append(npv)
        M["f1"].append(f1_score(yt, yp, zero_division=0))
        M["mcc"].append(matthews_corrcoef(yt, yp))
        M["auc"].append(stable_roc_auc(yt, pr))
        M["auprc"].append(stable_auprc(yt, pr))
        M["tn"].append(tn); M["fp"].append(fp); M["fn"].append(fn); M["tp"].append(tp)

    return M

def _logit(p: np.ndarray) -> np.ndarray:
    eps = 1e-12
    p = np.clip(p, eps, 1 - eps)
    return np.log(p) - np.log(1 - p)

def calibration_in_the_large(y_true: np.ndarray, y_prob: np.ndarray) -> float:
    y_true = as_numpy(y_true)
    z = _logit(as_numpy(y_prob)).reshape(-1, 1)
    lr = LogisticRegression(penalty=None, solver="lbfgs", max_iter=2000)
    lr.fit(z, y_true)
    return float(lr.intercept_[0])

def calibration_slope(y_true: np.ndarray, y_prob: np.ndarray) -> float:
    y_true = as_numpy(y_true)
    z = _logit(as_numpy(y_prob)).reshape(-1, 1)
    lr = LogisticRegression(penalty=None, solver="lbfgs", max_iter=2000)
    lr.fit(z, y_true)
    return float(lr.coef_[0][0])

def bootstrap_calibration(
    y_true: np.ndarray,
    y_prob: np.ndarray,
    groups: Iterable,
    n_boot: int = N_BOOTSTRAPS,
    seed: int = RANDOM_SEED,
) -> Dict[str, List[float]]:
    """Patient-cluster bootstrap for Brier score, CITL, and calibration slope."""
    rng = np.random.RandomState(seed)
    y_true = as_numpy(y_true)
    y_prob = as_numpy(y_prob)
    g_np = np.asarray(groups)

    if not (len(y_true) == len(y_prob) == len(g_np)):
        raise ValueError("y_true, y_prob, and groups must have identical lengths.")

    brier_vals, citl_vals, slope_vals = [], [], []

    for _ in range(n_boot):
        idx = _cluster_bootstrap_indices(g_np, rng)
        yt = y_true[idx]
        pr = y_prob[idx]

        brier_vals.append(float(brier_score_loss(yt, pr)))
        if len(np.unique(yt)) < 2:
            citl_vals.append(float("nan"))
            slope_vals.append(float("nan"))
            continue
        try:
            citl_vals.append(calibration_in_the_large(yt, pr))
            slope_vals.append(calibration_slope(yt, pr))
        except Exception:
            citl_vals.append(float("nan"))
            slope_vals.append(float("nan"))

    return {"brier": brier_vals, "citl": citl_vals, "slope": slope_vals}

def plot_and_save_roc(y_true: np.ndarray, y_prob: np.ndarray, name: str) -> None:
    y_true = as_numpy(y_true); y_prob = as_numpy(y_prob)
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    auc_val = stable_roc_auc(y_true, y_prob)

    plt.figure(figsize=(6.3, 4.6))
    plt.plot(fpr, tpr, label=f"AUC={auc_val:.3f}")
    plt.plot([0,1],[0,1],"--", linewidth=1)
    plt.xlabel("False positive rate")
    plt.ylabel("True positive rate")
    plt.title("ROC curve (CatBoost)")
    plt.legend(loc="lower right")
    plt.tight_layout()
    path = os.path.join(FIG_DIR, f"{name}_roc.png")
    plt.savefig(path)
    plt.show()
    plt.close()
    print(f"Saved: {path}")

def plot_and_save_pr(y_true: np.ndarray, y_prob: np.ndarray, name: str) -> None:
    y_true = as_numpy(y_true); y_prob = as_numpy(y_prob)
    prec, rec, _ = precision_recall_curve(y_true, y_prob)
    auprc_val = stable_auprc(y_true, y_prob)

    plt.figure(figsize=(6.3, 4.6))
    plt.plot(rec, prec, label=f"AUPRC={auprc_val:.3f}")
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title("Precision–Recall curve (CatBoost)")
    plt.legend(loc="lower left")
    plt.tight_layout()
    path = os.path.join(FIG_DIR, f"{name}_pr.png")
    plt.savefig(path)
    plt.show()
    plt.close()
    print(f"Saved: {path}")

def plot_and_save_calibration(y_true: np.ndarray, y_prob: np.ndarray, name: str, n_bins: int = 10) -> None:
    y_true = as_numpy(y_true); y_prob = as_numpy(y_prob)
    prob_true, prob_pred = calibration_curve(y_true, y_prob, n_bins=n_bins, strategy="uniform")

    plt.figure(figsize=(6.3, 4.6))
    plt.plot(prob_pred, prob_true, marker="o", linewidth=1, label="Model")
    plt.plot([0,1],[0,1],"--", linewidth=1, label="Perfect")
    plt.xlabel("Predicted probability")
    plt.ylabel("Observed frequency")
    plt.title("Calibration curve (CatBoost)")
    plt.legend()
    plt.tight_layout()
    path = os.path.join(FIG_DIR, f"{name}_calibration.png")
    plt.savefig(path)
    plt.show()
    plt.close()
    print(f"Saved: {path}")

def make_objective_optuna_catboost(
    X: pd.DataFrame,
    y: pd.Series,
    groups: Iterable,
    n_splits: int = N_SPLITS_INNER,
    seed: int = RANDOM_SEED,
):
    """Optuna objective: mean patient-grouped CV ROC AUC."""
    y_np = as_numpy(y)
    g_np = np.asarray(groups)
    if not (len(X) == len(y_np) == len(g_np)):
        raise ValueError("X, y, and groups must have identical lengths.")

    def objective(trial: optuna.trial.Trial) -> float:
        params = {
            "loss_function": "Logloss",
            "eval_metric": "AUC",

            # --- REQUIRED search space ---
            "depth": trial.suggest_int("depth", 3, 7),
            "iterations": trial.suggest_int("iterations", 1000, 2000),
            "learning_rate": trial.suggest_float("learning_rate", 0.05, 0.2, log=True),
            "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1e-2, 20.0, log=True),
            "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 5.0),
            "random_strength": trial.suggest_float("random_strength", 0.0, 2.0),
            "border_count": trial.suggest_int("border_count", 32, 255),

            # fixed/reproducibility
            "random_seed": seed,
            "verbose": False,
            "allow_writing_files": False,
            "thread_count": -1,
        }

        cv = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
        aucs = []

        for fold, (tr, va) in enumerate(cv.split(X, y_np, groups=g_np), start=1):
            overlap = np.intersect1d(np.unique(g_np[tr]), np.unique(g_np[va]))
            if overlap.size:
                raise RuntimeError(f"Patient overlap detected in Optuna fold {fold}.")
            m = CatBoostClassifier(**params)
            m.fit(Pool(X.iloc[tr], y_np[tr]))
            p = m.predict_proba(X.iloc[va])[:, 1]
            aucs.append(roc_auc_score(y_np[va], p))

        return float(np.mean(aucs))

    return objective

def build_catboost_from_params(params: Dict, seed: int = RANDOM_SEED) -> CatBoostClassifier:
    p = dict(params)
    p.update({
        "loss_function": "Logloss",
        "eval_metric": "AUC",
        "random_seed": seed,
        "verbose": False,
        "allow_writing_files": False,
        "thread_count": -1,
    })
    return CatBoostClassifier(**p)


def fit_calibrated_prefit(
    base_model,
    X_model: pd.DataFrame,
    y_model: np.ndarray,
    X_cal: pd.DataFrame,
    y_cal: np.ndarray,
    method: str,
) -> CalibratedClassifierCV:
    base_model.fit(Pool(X_model, y_model))
    cal = CalibratedClassifierCV(base_model, method=method, cv="prefit")
    cal.fit(X_cal, y_cal)
    return cal


def grouped_stratified_holdout_indices(
    X: pd.DataFrame,
    y: np.ndarray,
    groups: Iterable,
    holdout_frac: float = 0.2,
    seed: int = RANDOM_SEED,
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Create a patient-disjoint approximately stratified model/calibration split.
    A StratifiedGroupKFold split whose validation size is closest to holdout_frac is used.
    """
    y_np = as_numpy(y)
    g_np = np.asarray(groups)
    if not (len(X) == len(y_np) == len(g_np)):
        raise ValueError("X, y, and groups must have identical lengths.")
    if not 0 < holdout_frac < 1:
        raise ValueError("holdout_frac must be between 0 and 1.")

    n_unique = len(pd.unique(g_np))
    n_splits = int(round(1.0 / holdout_frac))
    n_splits = max(2, min(n_splits, n_unique))
    if n_splits < 2:
        raise ValueError("At least two unique patients are required for grouped holdout.")

    cv = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    candidates = list(cv.split(X, y_np, groups=g_np))
    if not candidates:
        raise RuntimeError("Could not create a grouped calibration holdout.")

    target_prev = float(np.mean(y_np))

    def score(split):
        tr, va = split
        size_penalty = abs(len(va) / len(X) - holdout_frac)
        prev_penalty = abs(float(np.mean(y_np[va])) - target_prev) if len(va) else 1.0
        return size_penalty + 0.25 * prev_penalty

    tr_idx, cal_idx = min(candidates, key=score)
    overlap = np.intersect1d(np.unique(g_np[tr_idx]), np.unique(g_np[cal_idx]))
    if overlap.size:
        raise RuntimeError("Patient overlap detected in grouped calibration holdout.")
    return np.asarray(tr_idx), np.asarray(cal_idx)


def get_oof_probabilities_catboost_with_optional_calibration(
    cat_params: Dict,
    X: pd.DataFrame,
    y: np.ndarray,
    groups: Iterable,
    calibration_mode: str = "uncal",  # "uncal" | "platt" | "iso"
    cal_holdout_frac: float = 0.2,
    n_splits: int = N_SPLITS_THRESHOLD,
    seed: int = RANDOM_SEED,
) -> np.ndarray:
    """
    Patient-grouped OOF probabilities on training for threshold selection.

    If calibration is requested, the OOF-training portion is additionally split
    into patient-disjoint model-fitting and calibration subsets.
    """
    y_np = as_numpy(y)
    g_np = np.asarray(groups)
    if not (len(X) == len(y_np) == len(g_np)):
        raise ValueError("X, y, and groups must have identical lengths.")

    rng = np.random.RandomState(seed)
    oof = np.full(shape=(len(X),), fill_value=np.nan, dtype=float)

    cv = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    for fold, (tr_idx, va_idx) in enumerate(cv.split(X, y_np, groups=g_np), start=1):
        overlap = np.intersect1d(np.unique(g_np[tr_idx]), np.unique(g_np[va_idx]))
        if overlap.size:
            raise RuntimeError(f"Patient overlap detected in threshold OOF fold {fold}.")

        X_tr, y_tr, g_tr = X.iloc[tr_idx], y_np[tr_idx], g_np[tr_idx]
        X_va = X.iloc[va_idx]
        base = build_catboost_from_params(cat_params, seed=seed)

        if calibration_mode == "uncal":
            base.fit(Pool(X_tr, y_tr))
            oof[va_idx] = base.predict_proba(X_va)[:, 1]

        elif calibration_mode in ("platt", "iso"):
            split_seed = int(rng.randint(0, 1_000_000))
            model_rel, cal_rel = grouped_stratified_holdout_indices(
                X_tr, y_tr, g_tr,
                holdout_frac=cal_holdout_frac,
                seed=split_seed,
            )
            X_model, y_model = X_tr.iloc[model_rel], y_tr[model_rel]
            X_cal, y_cal = X_tr.iloc[cal_rel], y_tr[cal_rel]

            method = "sigmoid" if calibration_mode == "platt" else "isotonic"
            cal = fit_calibrated_prefit(
                base, X_model, y_model, X_cal, y_cal, method=method
            )
            oof[va_idx] = cal.predict_proba(X_va)[:, 1]

        else:
            raise ValueError("calibration_mode must be 'uncal', 'platt', or 'iso'")

    if np.isnan(oof).any():
        raise RuntimeError("OOF probabilities contain NaNs; check data/CV.")
    return oof

def run_pipeline_catboost_external(
    X_train: pd.DataFrame,
    y_train: Iterable,
    groups_train: Iterable,
    X_external: pd.DataFrame,
    y_external: Iterable,
    groups_external: Iterable,
    feature_fill_strategy: str = "mean",
    n_trials: int = N_TRIALS,
    n_splits_inner: int = N_SPLITS_INNER,
    n_splits_threshold: int = N_SPLITS_THRESHOLD,
    n_bootstraps: int = N_BOOTSTRAPS,
    calibration_mode: str = "uncal",   # "uncal" | "platt" | "iso"
    cal_holdout_frac: float = 0.2,
    run_shap: bool = True,
    prefix: str = "cat_external",
) -> Dict[str, Dict[str, List[float]]]:
    """
    External validation with patient grouping in every development-side resampling
    step and patient-cluster bootstrap confidence intervals in the external cohort.
    """
    y_tr = as_numpy(y_train)
    y_ext = as_numpy(y_external)
    g_tr = np.asarray(groups_train)
    g_ext = np.asarray(groups_external)

    if not (len(X_train) == len(y_tr) == len(g_tr)):
        raise ValueError("Training X, y, and patient groups must have identical lengths.")
    if not (len(X_external) == len(y_ext) == len(g_ext)):
        raise ValueError("External X, y, and patient groups must have identical lengths.")

    # Patient/admission summaries
    for cohort_name, g in [("training", g_tr), ("external", g_ext)]:
        counts = pd.Series(g).value_counts()
        print(
            f"{cohort_name.capitalize()}: {len(g)} admissions, {len(counts)} unique patients, "
            f"{int((counts > 1).sum())} patients with recurrent admissions."
        )
        counts.rename("admissions").rename_axis("subject_reference").to_csv(
            os.path.join(OUT_DIR, f"{prefix}_{cohort_name}_admissions_per_patient.csv")
        )

    # --- 1) Tune CatBoost using patient-grouped CV
    print("Running Optuna hyperparameter tuning (CatBoost, patient-grouped CV AUC)...")
    study = optuna.create_study(direction="maximize")
    study.optimize(
        make_objective_optuna_catboost(
            X_train,
            pd.Series(y_tr),
            groups=g_tr,
            n_splits=n_splits_inner,
            seed=RANDOM_SEED,
        ),
        n_trials=n_trials,
        show_progress_bar=False,
    )
    best_params = study.best_params
    print("\nBest CatBoost params:", best_params)
    print(f"Best grouped CV AUC: {study.best_value:.4f}")
    pd.Series(best_params).to_json(os.path.join(OUT_DIR, f"{prefix}_best_params.json"))

    # --- 2) Threshold selection from patient-grouped TRAINING OOF probabilities
    print(
        f"\nComputing patient-grouped TRAINING OOF probabilities "
        f"for threshold selection (mode={calibration_mode})..."
    )
    p_oof = get_oof_probabilities_catboost_with_optional_calibration(
        cat_params=best_params,
        X=X_train,
        y=y_tr,
        groups=g_tr,
        calibration_mode=calibration_mode,
        cal_holdout_frac=cal_holdout_frac,
        n_splits=n_splits_threshold,
        seed=RANDOM_SEED,
    )
    thresholds = thresholds_from_predictions(y_tr, p_oof)
    thr_df = pd.DataFrame(
        {"rule": list(thresholds.keys()), "threshold": list(thresholds.values())}
    )
    thr_path = os.path.join(
        OUT_DIR, f"{prefix}_thresholds_from_train_oof_{calibration_mode}.csv"
    )
    thr_df.to_csv(thr_path, index=False)

    print("\nFrozen thresholds (patient-grouped TRAINING OOF):")
    display(thr_df)
    print(f"Saved: {thr_path}")

    # --- 3) Fit final model on training; if calibrated, keep patients disjoint
    X_ext = align_features(X_external, X_train, fill_strategy=feature_fill_strategy)

    if calibration_mode == "uncal":
        final_model = build_catboost_from_params(best_params, seed=RANDOM_SEED)
        final_model.fit(Pool(X_train, y_tr))
        p_ext = final_model.predict_proba(X_ext)[:, 1]
        fitted_for_shap = final_model
        X_shap_train = X_train

    elif calibration_mode in ("platt", "iso"):
        model_idx, cal_idx = grouped_stratified_holdout_indices(
            X_train,
            y_tr,
            g_tr,
            holdout_frac=cal_holdout_frac,
            seed=RANDOM_SEED,
        )
        overlap = np.intersect1d(
            np.unique(g_tr[model_idx]), np.unique(g_tr[cal_idx])
        )
        if overlap.size:
            raise RuntimeError("Patient overlap detected in final model/calibration split.")

        X_model, y_model = X_train.iloc[model_idx], y_tr[model_idx]
        X_cal, y_cal = X_train.iloc[cal_idx], y_tr[cal_idx]

        base = build_catboost_from_params(best_params, seed=RANDOM_SEED)
        method = "sigmoid" if calibration_mode == "platt" else "isotonic"
        final_cal = fit_calibrated_prefit(
            base, X_model, y_model, X_cal, y_cal, method=method
        )
        p_ext = final_cal.predict_proba(X_ext)[:, 1]

        # SHAP on the base CatBoost model (not the calibrator wrapper)
        fitted_for_shap = base
        X_shap_train = X_model

    else:
        raise ValueError("calibration_mode must be 'uncal', 'platt', or 'iso'")

    # --- 4) External curves
    print("\nExternal ROC/PR/Calibration plots:")
    plot_and_save_roc(y_ext, p_ext, name=f"{prefix}_{calibration_mode}")
    plot_and_save_pr(y_ext, p_ext, name=f"{prefix}_{calibration_mode}")
    plot_and_save_calibration(y_ext, p_ext, name=f"{prefix}_{calibration_mode}")

    # --- 5) External patient-cluster bootstrap metrics at fixed thresholds
    results: Dict[str, Dict[str, List[float]]] = {}
    summaries = []
    for rule, thr in thresholds.items():
        boot = bootstrap_metrics_fixed_threshold(
            y_ext,
            p_ext,
            groups=g_ext,
            threshold=thr,
            n_boot=n_bootstraps,
            seed=RANDOM_SEED,
        )
        results[rule] = boot

        df = summarize_bootstrap(boot)
        df.insert(0, "rule", rule)
        df.insert(1, "threshold", thr)
        df.insert(2, "calibration_mode", calibration_mode)
        summaries.append(df)

    perf_df = pd.concat(summaries, ignore_index=True)
    perf_path = os.path.join(
        OUT_DIR, f"{prefix}_external_cluster_bootstrap_metrics_{calibration_mode}.csv"
    )
    perf_df.to_csv(perf_path, index=False)

    print("\nExternal performance summary (patient-cluster bootstrap mean + 95% CI):")
    display(perf_df)
    print(f"Saved: {perf_path}")

    # --- 6) External calibration metrics
    print("\nCalibration analysis (External):")
    brier = float(brier_score_loss(y_ext, p_ext))
    citl = calibration_in_the_large(y_ext, p_ext)
    slope = calibration_slope(y_ext, p_ext)
    print(f"Brier score (point):       {brier:.4f}")
    print(f"CITL (point):              {citl:.4f}")
    print(f"Calibration slope (point): {slope:.4f}")

    C = bootstrap_calibration(
        y_ext,
        p_ext,
        groups=g_ext,
        n_boot=n_bootstraps,
        seed=RANDOM_SEED,
    )
    cal_rows = []
    for label, key in [("Brier", "brier"), ("CITL", "citl"), ("Slope", "slope")]:
        vals = [v for v in C[key] if not np.isnan(v)]
        m, lo, hi = ci_mean(vals)
        cal_rows.append([label, m, lo, hi, calibration_mode])

    cal_df = pd.DataFrame(
        cal_rows,
        columns=["metric", "mean", "ci_low", "ci_high", "calibration_mode"],
    )
    cal_path = os.path.join(
        OUT_DIR, f"{prefix}_external_calibration_cluster_bootstrap_{calibration_mode}.csv"
    )
    cal_df.to_csv(cal_path, index=False)
    display(cal_df)
    print(f"Saved: {cal_path}")

    # --- 7) SHAP (optional)
    if run_shap:
        print("\nRunning SHAP (TreeExplainer) on fitted CatBoost model...")
        run_shap_for_catboost(
            fitted_cat=fitted_for_shap,
            X_train_for_background=X_shap_train,
            X_external_aligned=X_ext,
            feature_names=list(X_train.columns),
            prefix=f"cat_{calibration_mode}",
        )

    print("\nDone.")
    return results

def run_shap_for_catboost(
    fitted_cat: CatBoostClassifier,
    X_train_for_background: pd.DataFrame,
    X_external_aligned: pd.DataFrame,
    feature_names: List[str],
    prefix: str = "cat",
    max_samples: int = 500,
    seed: int = RANDOM_SEED,
) -> None:
    try:
        import shap
    except ModuleNotFoundError:
        print("SHAP not installed. Install with: pip install shap")
        return

    def subsample_df(X: pd.DataFrame, n: int) -> pd.DataFrame:
        if len(X) <= n:
            return X
        rng = np.random.RandomState(seed)
        idx = rng.choice(len(X), n, replace=False)
        return X.iloc[idx]

    X_train_sub = subsample_df(X_train_for_background, max_samples)
    X_ext_sub = subsample_df(X_external_aligned, max_samples)

    explainer = shap.TreeExplainer(fitted_cat)
    shap_train = np.asarray(explainer.shap_values(X_train_sub))
    shap_ext = np.asarray(explainer.shap_values(X_ext_sub))

    # CatBoost sometimes includes an extra "bias" column at the end.
    def drop_bias_if_present(S: np.ndarray) -> np.ndarray:
        if S.ndim != 2:
            raise ValueError(f"Expected 2D SHAP matrix, got {S.shape}")
        if S.shape[1] == len(feature_names) + 1:
            return S[:, :-1]
        return S

    S_tr = drop_bias_if_present(shap_train)
    S_ext = drop_bias_if_present(shap_ext)

    def save_mean_abs(S: np.ndarray, name: str) -> str:
        mean_abs = np.abs(S).mean(axis=0)
        df = pd.DataFrame({"feature": feature_names, "mean_abs_shap": mean_abs}).sort_values(
            "mean_abs_shap", ascending=False
        )
        out = os.path.join(SHAP_DIR, f"{prefix}_shap_{name}.csv")
        df.to_csv(out, index=False)
        return out

    p1 = save_mean_abs(S_tr, "train")
    p2 = save_mean_abs(S_ext, "external")
    print(f"Saved: {p1}")
    print(f"Saved: {p2}")

    # Summary plots (saved + inline)
    plt.figure()
    shap.summary_plot(S_tr, X_train_sub, show=False)
    plt.tight_layout()
    out1 = os.path.join(FIG_DIR, f"{prefix}_shap_summary_train.png")
    plt.savefig(out1)
    plt.show()
    plt.close()
    print(f"Saved: {out1}")

    plt.figure()
    shap.summary_plot(S_ext, X_ext_sub, show=False)
    plt.tight_layout()
    out2 = os.path.join(FIG_DIR, f"{prefix}_shap_summary_external.png")
    plt.savefig(out2)
    plt.show()
    plt.close()
    print(f"Saved: {out2}")

# Requires: X_train, y_train, X_external, y_external

results_cat = run_pipeline_catboost_external(
    X_train=X_train,
    y_train=y_train,
    groups_train=subject_reference_train,
    X_external=X_external,
    y_external=y_external,
    groups_external=subject_reference_external,
    feature_fill_strategy="mean",
    n_trials=N_TRIALS,
    n_splits_inner=N_SPLITS_INNER,
    n_splits_threshold=N_SPLITS_THRESHOLD,
    n_bootstraps=N_BOOTSTRAPS,
    calibration_mode="platt",   # "uncal" / "platt" / "iso"
    cal_holdout_frac=0.2,
    run_shap=True,
    prefix="cat_external",
)
